In [1]:
# Importing all necessary libraries

import numpy as np
import pandas as pd

# Read the Excel file

In [25]:
data = pd.read_excel('course_recommendation.xlsx')

In [26]:
data.head()

,courseid,title,technology,topic,abstract,url,image_url
0,1,Categorical Data in the Tidyverse,R,Data Manipulation,NaN,https://www.datacamp.com/courses/categorical-d...,NaN
1,2,GARCH Models in R,R,Applied Finance,NaN,https://www.datacamp.com/courses/garch-models-...,NaN
2,3,Structural Equation Modeling with lavaan in R,R,Probability & Statistics,NaN,https://www.datacamp.com/courses/structural-eq...,NaN
3,4,Analyzing IoT Data in Python,Python,Data Manipulation,NaN,https://www.datacamp.com/courses/analyzing-iot...,NaN
4,5,Introduction to Spark SQL in Python,Python,Data Manipulation,NaN,https://www.datacamp.com/courses/introduction-...,NaN


#### Description of the data
## The file contains 4 features about the course which have been uploaded into our database
* title - Title of the Course
* technology - Technology of the course 
* topic - course topic
* abstract - Short summary of the course
* url - Course url - if applicable

In [27]:
data.isnull().sum()

courseid        0
title           0
technology      0
topic         154
abstract      272
url             0
image_url     273
dtype: int64

In [28]:
data.fillna(" ",inplace=True)

In [29]:
data.isnull().sum()

courseid      0
title         0
technology    0
topic         0
abstract      0
url           0
image_url     0
dtype: int64

# Objective of the Project
* Recommend top 10 courses (course name and course link ) based on the user's browse history / search history 
* The 'tags' field is used to fetch the similar courses 

In [31]:
# The 4  columns 'technology' , topic' and 'abstract'  contain important keywords that the user might be using to search the course. Hence three fields are combined together to create a new field 'tags' to help us to recommend the appropriate courses 
data['tag'] = data['title'] + " " + data['technology'] +" " + data['topic'] + " "+data['abstract']

In [32]:
data.head()

,courseid,title,technology,topic,abstract,url,image_url,tag
0,1,Categorical Data in the Tidyverse,R,Data Manipulation,,https://www.datacamp.com/courses/categorical-d...,,Categorical Data in the Tidyverse R Data Manip...
1,2,GARCH Models in R,R,Applied Finance,,https://www.datacamp.com/courses/garch-models-...,,GARCH Models in R R Applied Finance
2,3,Structural Equation Modeling with lavaan in R,R,Probability & Statistics,,https://www.datacamp.com/courses/structural-eq...,,Structural Equation Modeling with lavaan in R ...
3,4,Analyzing IoT Data in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/analyzing-iot...,,Analyzing IoT Data in Python Python Data Manip...
4,5,Introduction to Spark SQL in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/introduction-...,,Introduction to Spark SQL in Python Python Dat...


In [33]:
# Required fields - title, tag and url
# All other fields can be dropped

df = data[['title','tag','url']]

In [34]:
df.head()

,title,tag,url
0,Categorical Data in the Tidyverse,Categorical Data in the Tidyverse R Data Manip...,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,GARCH Models in R R Applied Finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,Structural Equation Modeling with lavaan in R ...,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,Analyzing IoT Data in Python Python Data Manip...,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,Introduction to Spark SQL in Python Python Dat...,https://www.datacamp.com/courses/introduction-...


# Text Preprocessing

In [38]:
# Convert all tags into lowercase

df['tag'] = df['tag'].apply(lambda x:x.lower())

C:\Users\Suhas\AppData\Local\Temp\ipykernel_24028\2192981827.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tag'] = df['tag'].apply(lambda x:x.lower())


In [41]:
df.head()

,title,tag,url
0,Categorical Data in the Tidyverse,categorical data in the tidyverse r data manip...,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,garch models in r r applied finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,structural equation modeling with lavaan in r ...,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,analyzing iot data in python python data manip...,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,introduction to spark sql in python python dat...,https://www.datacamp.com/courses/introduction-...


In [46]:
# Data cleaning - Removing all punctuation marks and stop words from tags 

import nltk
import string

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

def data_clean(text):
    result = []
    words = nltk.word_tokenize(text)
    for word in words:
        if (word not in string.punctuation) & (word not in stopwords.words('english')):
            result.append(word)
    return " ".join(result)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Suhas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Suhas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Suhas\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [47]:
df['tag'] = df['tag'].apply(data_clean)

C:\Users\Suhas\AppData\Local\Temp\ipykernel_24028\2791499102.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tag'] = df['tag'].apply(data_clean)


In [48]:
df.head()

,title,tag,url
0,Categorical Data in the Tidyverse,categorical data tidyverse r data manipulation,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,garch models r r applied finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,structural equation modeling lavaan r r probab...,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,analyzing iot data python python data manipula...,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,introduction spark sql python python data mani...,https://www.datacamp.com/courses/introduction-...


### Count vectorizer:


In [49]:
# Vectorization - converting 'tags' columns into vectors
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

tags_vectors = cv.fit_transform(df['tag'])


In [50]:
# Similarity measure - Calculating the similarity of tags of each course with itself and all other courses 
# This is used to find how similar the courses are with each other

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tags_vectors)

In [51]:
similarity

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.01891512,
        0.05801194],
       [0.        , 0.        , 0.        , ..., 0.01891512, 1.        ,
        0.22416327],
       [0.        , 0.        , 0.        , ..., 0.05801194, 0.22416327,
        1.        ]])

In [90]:
def recommend_course(course):
    index = df[df['title']==course].index[0]
    distances = similarity[index]
    course_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:11]
    #print(course_list)
    
    for i in course_list:
        print(df.iloc[i[0]].title," --", df.iloc[i[0]].url)
        #print(df.iloc[i[0]].title,"\n")

In [91]:
recommend_course('Analyzing IoT Data in Python')

Analyzing Social Media Data in Python  -- https://www.datacamp.com/courses/analyzing-social-media-data-in-python
Working with Geospatial Data in Python  -- https://www.datacamp.com/courses/working-with-geospatial-data-in-python
Dealing with Missing Data in Python  -- https://www.datacamp.com/courses/dealing-with-missing-data-in-python
Manipulating Time Series Data in Python  -- https://www.datacamp.com/courses/manipulating-time-series-data-in-python
Introduction to Databases in Python  -- https://www.datacamp.com/courses/introduction-to-relational-databases-in-python
Introduction to MongoDB in Python  -- https://www.datacamp.com/courses/introduction-to-using-mongodb-for-data-science-with-python
Regular Expressions in Python  -- https://www.datacamp.com/courses/regular-expressions-in-python
Visualizing Geospatial Data in Python  -- https://www.datacamp.com/courses/visualizing-geospatial-data-in-python
Improving Your Data Visualizations in Python  -- https://www.datacamp.com/courses/impr

In [92]:
recommend_course("Introduction to Spark SQL in Python")

Introduction to Databases in Python  -- https://www.datacamp.com/courses/introduction-to-relational-databases-in-python
Introduction to MongoDB in Python  -- https://www.datacamp.com/courses/introduction-to-using-mongodb-for-data-science-with-python
Introduction to Data Science in Python  -- https://www.datacamp.com/courses/introduction-to-data-science-in-python
Regular Expressions in Python  -- https://www.datacamp.com/courses/regular-expressions-in-python
Analyzing IoT Data in Python  -- https://www.datacamp.com/courses/analyzing-iot-data-in-python
Working with Geospatial Data in Python  -- https://www.datacamp.com/courses/working-with-geospatial-data-in-python
Dealing with Missing Data in Python  -- https://www.datacamp.com/courses/dealing-with-missing-data-in-python
Analyzing Social Media Data in Python  -- https://www.datacamp.com/courses/analyzing-social-media-data-in-python
Manipulating Time Series Data in Python  -- https://www.datacamp.com/courses/manipulating-time-series-data

In [93]:
recommend_course("Categorical Data in the Tidyverse")

Working with Data in the Tidyverse  -- https://www.datacamp.com/courses/working-with-data-in-the-tidyverse
Joining Data in R with data.table  -- https://www.datacamp.com/courses/joining-data-in-r-with-datatable
Data Manipulation with dplyr in R  -- https://www.datacamp.com/courses/data-manipulation-with-dplyr-in-r
Communicating with Data in the Tidyverse  -- https://www.datacamp.com/courses/communicating-with-data-in-the-tidyverse
Data Manipulation in R with data.table  -- https://www.datacamp.com/courses/data-manipulation-in-r-with-datatable
Joining Data with dplyr in R  -- https://www.datacamp.com/courses/joining-data-with-dplyr-in-r
PostgreSQL Functions for Manipulating Data  -- https://www.datacamp.com/courses/postgresql-functions-for-manipulating-data
Joining Data in SQL  -- https://www.datacamp.com/courses/joining-data-in-postgresql
Data Processing in Shell  -- https://www.datacamp.com/courses/data-processing-in-shell
Working with Geospatial Data in R  -- https://www.datacamp.com

In [94]:
recommend_course("how to tackle gre")

GRE Prep: Crack the Quantitative Reasoning Section  -- https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=507388.725198&type=2&murl=https%3A%2F%2Fwww.udemy.com%2Fcourse%2Fgre-prep-crack-the-quantitative-reasoning-section%2F
Improve your GRE score  -- https://imp.i154272.net/c/2411299/826178/11490
Reading Comprehension for SAT, GMAT, GRE, CAT  -- https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=507388.1803754&type=2&murl=https%3A%2F%2Fwww.udemy.com%2Fcourse%2Freading-comprehension-practice%2F
English Vocabulary - SAT, GRE, GMAT, TOEFL  -- https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=507388.1518478&type=2&murl=https%3A%2F%2Fwww.udemy.com%2Fcourse%2Fdontmemorise_vocabulary%2F
Vocabulary for SAT, GMAT, GRE, CAT applicants  -- https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=507388.1807772&type=2&murl=https%3A%2F%2Fwww.udemy.com%2Fcourse%2Fvocabulary-practice-for-entrance-exams%2F
Calculation tricks  -- https://click.linksynergy.com/link?id=16zYSsTLZqk&

# Conclusion

## Cosine Similarity (Count Vectorizer)  method recommends courses correctly